# Is the Judge the Bottleneck? Kaggle notebook

Run this notebook **top to bottom** at the start of every Kaggle session. Kaggle forgets everything between sessions, so each run sets the project up again.

**Before you press Run, set these in the right-hand panel (Session options):**
1. **Accelerator:** `GPU T4 x2` (needed for sections 9–11; sections 1–8 also work on CPU).
2. **Internet:** On (needs a phone-verified Kaggle account).
3. **Add Input → Your Datasets → `hotpotqa-bm25-index`** (our saved search index).

Every section stops with a clear message if something goes wrong. Long lists of yellow "dependency conflict" warnings during installs are normal on Kaggle and can be ignored.

## 1. Settings

**What it does:** the few choices you might change: which GitHub branch to use, and which optional sections to run.
**Files used:** none.

In [ ]:
REPO_URL = "https://github.com/shoimya/Is-the-Judge-the-Bottleneck.git"
BRANCH = "SC"                     # the GitHub branch to run; "main" once work is merged

BUILD_INDEX_IF_MISSING = False    # True only if the hotpotqa-bm25-index dataset is lost (rebuild takes ~40 min)
RUN_MODEL_CHECKS = True           # sections 9-11: install vLLM, test the model, measure speed (needs GPU T4 x2)

## 2. Helpers and machine check

**What it does:** defines `run()`, which runs a command and **stops the notebook if it fails** (instead of carrying on into confusing errors). Then it checks this machine: Kaggle or Colab, Python version, GPU, and internet.
**Files used:** none.

In [ ]:
import os
import shutil
import socket
import sys

from IPython import get_ipython


def run(command: str) -> None:
    """Run a shell command, show its output, and stop the notebook with a clear message if it fails."""
    get_ipython().system(command)
    exit_code = get_ipython().user_ns.get("_exit_code", 0)
    if exit_code != 0:
        raise RuntimeError(f"This step failed (exit code {exit_code}): {command}\nSee the output above.")


IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
WORKING_DIR = "/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else os.getcwd()
REPO_DIR = os.path.join(WORKING_DIR, "Is-the-Judge-the-Bottleneck")
HAS_GPU = shutil.which("nvidia-smi") is not None

print("Machine:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local")
print("Python: ", sys.version.split()[0], "(the Mac uses 3.12; Kaggle may be newer, which is fine)")
print("GPU:    ", "yes" if HAS_GPU else "none (set Accelerator to GPU T4 x2 for sections 9-11)")
if HAS_GPU:
    run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

try:
    socket.gethostbyname("github.com")
    print("Internet: on")
except OSError:
    raise SystemExit("No internet. On Kaggle: verify your phone number (kaggle.com > Settings), "
                     "then switch Internet on in the notebook's settings panel and run all again.")

## 3. Get the project code

**What it does:** downloads our code from GitHub into Kaggle's working folder. If it's already there (re-running in the same session), it switches to `BRANCH` and pulls the latest version instead.
**Files used:** the GitHub repository `shoimya/Is-the-Judge-the-Bottleneck`, branch set in section 1.

In [ ]:
if os.path.isdir(REPO_DIR):
    run(f"git -C {REPO_DIR} fetch origin")
    run(f"git -C {REPO_DIR} checkout {BRANCH}")
    run(f"git -C {REPO_DIR} pull --ff-only origin {BRANCH}")
else:
    run(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
run("git log -1 --oneline")   # shows which commit this session is running

## 4. Install the project's Python libraries

**What it does:** installs the exact library versions the project needs (search, data download, config reading).
**Files used:** `requirements.txt` from the repository.
**Expected:** a long list of yellow "dependency conflicts" about Kaggle's own packages (Google Cloud, RAPIDS, gradio...). We don't use those; ignore the list. Only a red error that stops the cell matters.

In [ ]:
run("pip install -q -r requirements.txt")

## 5. Keep run logs safe

**What it does:** points the project's `runs/` folder at storage that survives the session (Kaggle: `/kaggle/working/runs`, saved when you click *Save Version*; Colab: Google Drive).
**Files used:** none yet; later tickets write run logs here.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    runs_storage = "/content/drive/MyDrive/Is-the-Judge-the-Bottleneck/runs"
else:
    runs_storage = os.path.join(WORKING_DIR, "runs")

os.makedirs(runs_storage, exist_ok=True)
if not os.path.islink("runs"):
    shutil.rmtree("runs", ignore_errors=True)
    os.symlink(runs_storage, "runs")
print("runs/ ->", os.path.realpath("runs"))

## 6. Setup check

**What it does:** runs the project's own check: Python version, settings file, and that `runs/` can be written. It must end with **`Setup OK.`** A warning that "the project targets Python 3.12" is expected on Kaggle's newer Python.
**Files used:** `run.py` and `config.yaml` from the repository.

In [ ]:
run("python run.py --check")

## 7. Question sets (T02)

**What it does:** downloads HotpotQA's dev questions and picks the same 100 Pilot and 1,000 Test questions as on the Mac (fixed seed), then confirms the picked question ids match the lists saved in GitHub.
**Files used:** downloaded from Hugging Face (`hotpotqa/hotpot_qa`, 28 MB) into `data/hotpotqa/`; compared against `results/question_ids/` from the repository.

In [ ]:
run("python -m pipeline.dataset")
run("git diff --exit-code --stat results/question_ids")   # fails if the questions differ from the Mac's
print("Same 100 + 1,000 questions as on the Mac.")

## 8. Search index (T03)

**What it does:** connects our saved BM25 search index to the project, then searches with the 100 Pilot questions and checks the results match the Mac's (11% / 24% / 34% / 44% "both Gold paragraphs found").
**Files used:** the attached Kaggle dataset `hotpotqa-bm25-index` (7 files, ~2.8 GB, read-only under `/kaggle/input/`), linked as `data/wiki/bm25_index`; compared against `results/pilot/bm25_recall.csv` from the repository.
If the dataset isn't attached, attach it (Add Input) and re-run this cell. Only if it's lost for good, set `BUILD_INDEX_IF_MISSING = True` in section 1: that downloads Wikipedia (1.55 GB) from Stanford and rebuilds (~40 min).

In [ ]:
import glob

index_link = "data/wiki/bm25_index"
attached_index = glob.glob("/kaggle/input/**/bm25_index/params.index.json", recursive=True)

if os.path.exists(index_link):
    print("Index already linked.")
elif attached_index:
    os.makedirs("data/wiki", exist_ok=True)
    os.symlink(os.path.dirname(attached_index[0]), index_link)
    print("Linked the attached index:", os.path.dirname(attached_index[0]))
elif BUILD_INDEX_IF_MISSING:
    run("python -m pipeline.retriever build")
    run("rm -rf data/wiki/dump")   # keep only the ~3 GB index in the output
else:
    raise SystemExit("Search index not found. Attach the hotpotqa-bm25-index dataset "
                     "(Add Input > Your Datasets) and re-run this cell.")

run("python -m pipeline.retriever check")
run("git diff --exit-code results/pilot/bm25_recall.csv")   # fails if search results differ from the Mac's
print("Search results match the Mac.")

## 9. Install vLLM (T04, GPU only)

**What it does:** installs vLLM, the program that runs the AI model on the GPU. It then removes TorchAudio, an audio add-on Kaggle preinstalls that is built for a different CUDA version than vLLM's PyTorch and otherwise crashes vLLM. We never use audio. Finally it prints the key versions.
**Files used:** `requirements-gpu.txt` from the repository (vLLM 0.30.0, ~5-10 min to install).

In [ ]:
if not RUN_MODEL_CHECKS:
    print("Skipped (RUN_MODEL_CHECKS is False).")
elif not HAS_GPU:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in Session options, then run all again.")
else:
    run("pip install -q -r requirements-gpu.txt")
    run("pip uninstall -y -q torchaudio")
    run("python -c \"import torch, vllm, transformers, huggingface_hub; "
        "print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'GPU available:', torch.cuda.is_available()); "
        "print('vllm', vllm.__version__, '| transformers', transformers.__version__, "
        "'| huggingface_hub', huggingface_hub.__version__)\"")

## 10. Model smoke test (T04, GPU only)

**What it does:** loads the AI model and asks three short questions, one per role (answerer, rewriter, judge). For each it prints the answer, token counts, and the model's top alternatives for its last word. The judge's line shows how sure it was of "no" versus "yes".
**Files used:** the model `Qwen/Qwen3-4B-Instruct-2507`, downloaded from Hugging Face (~8 GB, into Kaggle's storage, not your Mac).

In [ ]:
if RUN_MODEL_CHECKS:
    run("python -m pipeline.llm smoke --backend vllm")

## 11. Speed benchmark (T04, GPU only)

**What it does:** runs 20 Pilot questions through a 3-Round practice loop using real search results, and measures how fast the model processes text. The last four printed lines are the results: **copy them into the Throughput section of `NOTES.md`** (or paste them to Claude).
**Files used:** the Pilot questions (section 7), the search index (section 8) and the model (section 10, reused from Kaggle's cache).

In [ ]:
if RUN_MODEL_CHECKS:
    run("python -m pipeline.llm benchmark --backend vllm")